# 00 Init

In [3]:
import os
from dataclasses import dataclass
from typing import Dict, List, Any

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

# 01 Mock Data

In [4]:
# 单商家 mock 数据
merchant_df = pd.DataFrame([
    {
        "merchant_id": "M001",
        "merchant_name": "Blue Bottle Demo Store",
        "category": "coffee",
        "district": "Pudong",
        "orders_7d": 120,
        "orders_prev_7d": 145,
        "orders_last_30d_pctl": 0.94,  # 历史异常分位（MVP 先 mock）
        "orders_peer_pctl": 0.78,      # 同行异常分位（MVP 先 mock）
        "impressions": 5200,
        "conversion_rate": 0.021,
        "image_coverage": 0.46,
        "open_hours": 8.5,
        "accept_time_mins": 3.2,
        "prep_time_mins": 18.0,
        "merchant_cancel_rate": 0.075,
        "active_spu_count": 22,
        "discount_rate": 0.06,
    },
    {
        "merchant_id": "M002",
        "merchant_name": "Sunrise Burger Demo Store",
        "category": "burger",
        "district": "Xuhui",
        "orders_7d": 280,
        "orders_prev_7d": 270,
        "orders_last_30d_pctl": 0.55,
        "orders_peer_pctl": 0.42,
        "impressions": 10200,
        "conversion_rate": 0.034,
        "image_coverage": 0.81,
        "open_hours": 12.0,
        "accept_time_mins": 2.1,
        "prep_time_mins": 12.5,
        "merchant_cancel_rate": 0.021,
        "active_spu_count": 41,
        "discount_rate": 0.11,
    }
])

# 同类商家 benchmark 分布（MVP 用随机数模拟）
np.random.seed(42)

peer_benchmark = {
    "conversion_rate": np.clip(np.random.normal(0.03, 0.005, 200), 0.005, 0.2),
    "image_coverage": np.clip(np.random.normal(0.78, 0.12, 200), 0.05, 1.0),
    "open_hours": np.clip(np.random.normal(11.0, 1.8, 200), 4.0, 20.0),
    "accept_time_mins": np.clip(np.random.normal(2.5, 0.6, 200), 0.5, 10.0),
    "prep_time_mins": np.clip(np.random.normal(14.0, 2.5, 200), 5.0, 40.0),
    "merchant_cancel_rate": np.clip(np.random.normal(0.03, 0.012, 200), 0.0, 0.25),
    "active_spu_count": np.clip(np.random.normal(35, 8, 200), 1, 200),
    "discount_rate": np.clip(np.random.normal(0.10, 0.03, 200), 0.0, 0.6),
}

merchant_df.head()

,merchant_id,merchant_name,category,district,orders_7d,orders_prev_7d,orders_last_30d_pctl,orders_peer_pctl,impressions,conversion_rate,image_coverage,open_hours,accept_time_mins,prep_time_mins,merchant_cancel_rate,active_spu_count,discount_rate
0,M001,Blue Bottle Demo Store,coffee,Pudong,120,145,0.94,0.78,5200,0.021,0.46,8.5,3.2,18.0,0.075,22,0.06
1,M002,Sunrise Burger Demo Store,burger,Xuhui,280,270,0.55,0.42,10200,0.034,0.81,12.0,2.1,12.5,0.021,41,0.11


# 02 Inputs

In [5]:
merchant_id = "M001"
question = "这家商家最近经营怎么样？我应该建议他做什么？"

merchant_row = merchant_df.loc[merchant_df["merchant_id"] == merchant_id].iloc[0]
merchant_row

merchant_id                               M001
merchant_name           Blue Bottle Demo Store
category                                coffee
district                                Pudong
orders_7d                                  120
orders_prev_7d                             145
orders_last_30d_pctl                      0.94
orders_peer_pctl                          0.78
impressions                               5200
conversion_rate                          0.021
image_coverage                            0.46
open_hours                                 8.5
accept_time_mins                           3.2
prep_time_mins                            18.0
merchant_cancel_rate                     0.075
active_spu_count                            22
discount_rate                             0.06
Name: 0, dtype: object

# 03 Configs

In [6]:
HEALTH_CONFIG = {
    "warning_threshold": 0.70,
    "risk_hist_threshold": 0.90,
    "risk_peer_threshold": 0.70,
}

GAP_CONFIG = {
    "gap_percentile_threshold": 0.30,
    "strong_gap_percentile_threshold": 0.15,
}

# 展示顺序固定
DISPLAY_ORDER = [
    "traffic",
    "ops_readiness",
    "user_experience",
    "supply_quality",
]

# 决策权重动态
DRIVER_WEIGHTS = {
    "traffic": 0.35,
    "ops_readiness": 0.25,
    "user_experience": 0.20,
    "supply_quality": 0.20,
}

METRIC_TO_DRIVER = {
    "conversion_rate": "traffic",
    "image_coverage": "ops_readiness",
    "open_hours": "ops_readiness",
    "accept_time_mins": "ops_readiness",
    "prep_time_mins": "user_experience",
    "merchant_cancel_rate": "user_experience",
    "active_spu_count": "supply_quality",
    "discount_rate": "supply_quality",
}

ACTION_MAP = {
    "conversion_rate": "优化商品图片或设置更有吸引力的折扣",
    "image_coverage": "补齐菜单图片，优先补热销商品图",
    "open_hours": "延长晚间营业时间，覆盖高峰时段",
    "accept_time_mins": "优化接单流程，缩短接单时长",
    "prep_time_mins": "优化后厨流程，缩短出餐时间",
    "merchant_cancel_rate": "减少商责取消，优化备货与接单判断",
    "active_spu_count": "补充动销 SKU，优化菜单供给结构",
    "discount_rate": "提升核心商品折扣力度，增强价格竞争力",
}

# True = 越大越好；False = 越小越好
METRIC_DIRECTION = {
    "conversion_rate": True,
    "image_coverage": True,
    "open_hours": True,
    "accept_time_mins": False,
    "prep_time_mins": False,
    "merchant_cancel_rate": False,
    "active_spu_count": True,
    "discount_rate": True,
}

# 04 Data sturcture

In [7]:
@dataclass
class HealthResult:
    status: str
    hist_percentile: float
    peer_percentile: float
    summary: str


@dataclass
class GapResult:
    metric: str
    driver: str
    percentile: float
    value: float
    label: str


@dataclass
class ActionResult:
    driver: str
    metric: str
    action: str
    score: float
    priority: str

# 05 Helper Functions

In [8]:
def percentile_rank(value: float, population: np.ndarray, higher_is_better: bool = True) -> float:
    """
    返回 value 在 population 中的相对分位（0~1）。
    对于 higher_is_better=False 的指标，会自动反转，使得“表现越差 → percentile 越低”。
    """
    population = np.asarray(population)
    raw = float((population < value).mean())
    return raw if higher_is_better else 1 - raw


def gap_severity(percentile: float) -> float:
    """
    把 percentile 映射成 gap 严重程度。
    percentile 越低，severity 越高。
    """
    return max(0.0, 1.0 - percentile)


def priority_label(score: float) -> str:
    if score >= 0.60:
        return "High"
    if score >= 0.35:
        return "Medium"
    return "Low"

# 06 Health Model

In [9]:
def calculate_growth_health(row: pd.Series, config: Dict[str, float]) -> HealthResult:
    hist_p = float(row["orders_last_30d_pctl"])
    peer_p = float(row["orders_peer_pctl"])

    if hist_p >= config["risk_hist_threshold"] and peer_p >= config["risk_peer_threshold"]:
        status = "Risk"
    elif hist_p >= config["warning_threshold"] or peer_p >= config["warning_threshold"]:
        status = "Warning"
    else:
        status = "Healthy"

    growth = (row["orders_7d"] - row["orders_prev_7d"]) / row["orders_prev_7d"]
    summary = (
        f"近7日订单变化 {growth:.1%}；"
        f"历史异常分位 {hist_p:.0%}；"
        f"同行异常分位 {peer_p:.0%}。"
    )

    return HealthResult(
        status=status,
        hist_percentile=hist_p,
        peer_percentile=peer_p,
        summary=summary,
    )


health_result = calculate_growth_health(merchant_row, HEALTH_CONFIG)
health_result

HealthResult(status='Risk', hist_percentile=0.94, peer_percentile=0.78, summary='近7日订单变化 -17.2%；历史异常分位 94%；同行异常分位 78%。')

# 07 Gap model

In [10]:
def identify_gaps(
    row: pd.Series,
    peer_data: Dict[str, np.ndarray],
    gap_config: Dict[str, float],
) -> List[GapResult]:
    gap_results: List[GapResult] = []

    for metric, driver in METRIC_TO_DRIVER.items():
        value = float(row[metric])
        higher_is_better = METRIC_DIRECTION[metric]
        p = percentile_rank(value, peer_data[metric], higher_is_better=higher_is_better)

        if p < gap_config["strong_gap_percentile_threshold"]:
            label = "Strong Gap"
        elif p < gap_config["gap_percentile_threshold"]:
            label = "Gap"
        elif p > 0.70:
            label = "Strong"
        else:
            label = "Normal"

        gap_results.append(
            GapResult(
                metric=metric,
                driver=driver,
                percentile=p,
                value=value,
                label=label,
            )
        )

    return gap_results


gap_results = identify_gaps(merchant_row, peer_benchmark, GAP_CONFIG)
pd.DataFrame([g.__dict__ for g in gap_results]).sort_values(["driver", "percentile"])

,metric,driver,percentile,value,label
1,image_coverage,ops_readiness,0.005,0.460,Strong Gap
2,open_hours,ops_readiness,0.100,8.500,Strong Gap
3,accept_time_mins,ops_readiness,0.110,3.200,Strong Gap
6,active_spu_count,supply_quality,0.035,22.000,Strong Gap
7,discount_rate,supply_quality,0.080,0.060,Strong Gap
0,conversion_rate,traffic,0.025,0.021,Strong Gap
5,merchant_cancel_rate,user_experience,0.000,0.075,Strong Gap
4,prep_time_mins,user_experience,0.060,18.000,Strong Gap


# 08 Action Engine

In [11]:
def generate_actions(gaps: List[GapResult]) -> List[ActionResult]:
    actions: List[ActionResult] = []

    for gap in gaps:
        if gap.label not in {"Gap", "Strong Gap"}:
            continue

        severity = gap_severity(gap.percentile)
        score = severity * DRIVER_WEIGHTS[gap.driver]

        actions.append(
            ActionResult(
                driver=gap.driver,
                metric=gap.metric,
                action=ACTION_MAP[gap.metric],
                score=score,
                priority=priority_label(score),
            )
        )

    actions = sorted(actions, key=lambda x: x.score, reverse=True)
    return actions


action_results = generate_actions(gap_results)
pd.DataFrame([a.__dict__ for a in action_results])

,driver,metric,action,score,priority
0,traffic,conversion_rate,优化商品图片或设置更有吸引力的折扣,0.34125,Low
1,ops_readiness,image_coverage,补齐菜单图片，优先补热销商品图,0.24875,Low
2,ops_readiness,open_hours,延长晚间营业时间，覆盖高峰时段,0.22500,Low
3,ops_readiness,accept_time_mins,优化接单流程，缩短接单时长,0.22250,Low
4,user_experience,merchant_cancel_rate,减少商责取消，优化备货与接单判断,0.20000,Low
5,supply_quality,active_spu_count,补充动销 SKU，优化菜单供给结构,0.19300,Low
6,user_experience,prep_time_mins,优化后厨流程，缩短出餐时间,0.18800,Low
7,supply_quality,discount_rate,提升核心商品折扣力度，增强价格竞争力,0.18400,Low


# 09 Sturctured output

In [12]:
def build_structured_output(
    merchant: pd.Series,
    health: HealthResult,
    gaps: List[GapResult],
    actions: List[ActionResult],
    question: str,
) -> Dict[str, Any]:
    grouped_gaps: Dict[str, List[Dict[str, Any]]] = {driver: [] for driver in DISPLAY_ORDER}

    for g in gaps:
        grouped_gaps[g.driver].append({
            "metric": g.metric,
            "value": g.value,
            "percentile": round(g.percentile, 3),
            "label": g.label,
        })

    return {
        "merchant_id": merchant["merchant_id"],
        "merchant_name": merchant["merchant_name"],
        "question": question,
        "health": health.__dict__,
        "gaps_by_driver": grouped_gaps,
        "recommended_actions": [a.__dict__ for a in actions],
    }


structured_output = build_structured_output(
    merchant=merchant_row,
    health=health_result,
    gaps=gap_results,
    actions=action_results,
    question=question,
)

structured_output

{'merchant_id': 'M001',
 'merchant_name': 'Blue Bottle Demo Store',
 'question': '这家商家最近经营怎么样？我应该建议他做什么？',
 'health': {'status': 'Risk',
  'hist_percentile': 0.94,
  'peer_percentile': 0.78,
  'summary': '近7日订单变化 -17.2%；历史异常分位 94%；同行异常分位 78%。'},
 'gaps_by_driver': {'traffic': [{'metric': 'conversion_rate',
    'value': 0.021,
    'percentile': 0.025,
    'label': 'Strong Gap'}],
  'ops_readiness': [{'metric': 'image_coverage',
    'value': 0.46,
    'percentile': 0.005,
    'label': 'Strong Gap'},
   {'metric': 'open_hours',
    'value': 8.5,
    'percentile': 0.1,
    'label': 'Strong Gap'},
   {'metric': 'accept_time_mins',
    'value': 3.2,
    'percentile': 0.11,
    'label': 'Strong Gap'}],
  'user_experience': [{'metric': 'prep_time_mins',
    'value': 18.0,
    'percentile': 0.06,
    'label': 'Strong Gap'},
   {'metric': 'merchant_cancel_rate',
    'value': 0.075,
    'percentile': 0.0,
    'label': 'Strong Gap'}],
  'supply_quality': [{'metric': 'active_spu_count',
    'value'

# 10 Human Readable Output

In [13]:
def render_report(output: Dict[str, Any]) -> None:
    print("=== Merchant Growth Copilot ===")
    print(f"Merchant: {output['merchant_name']} ({output['merchant_id']})")
    print(f"Question: {output['question']}")
    print()

    print("[1] Health Status")
    print(f"- Status: {output['health']['status']}")
    print(f"- Summary: {output['health']['summary']}")
    print()

    print("[2] Key Gaps by Driver")
    for driver in DISPLAY_ORDER:
        print(f"- {driver}")
        items = output["gaps_by_driver"][driver]
        if not items:
            print("  - No metrics")
            continue

        for item in items:
            print(
                f"  - {item['metric']}: {item['label']} "
                f"(value={item['value']}, percentile={item['percentile']:.1%})"
            )
    print()

    print("[3] Recommended Actions")
    if not output["recommended_actions"]:
        print("- No recommended actions")
    else:
        for action in output["recommended_actions"]:
            print(
                f"- [{action['priority']}] {action['action']} "
                f"(driver={action['driver']}, metric={action['metric']}, score={action['score']:.2f})"
            )


render_report(structured_output)

=== Merchant Growth Copilot ===
Merchant: Blue Bottle Demo Store (M001)
Question: 这家商家最近经营怎么样？我应该建议他做什么？

[1] Health Status
- Status: Risk
- Summary: 近7日订单变化 -17.2%；历史异常分位 94%；同行异常分位 78%。

[2] Key Gaps by Driver
- traffic
  - conversion_rate: Strong Gap (value=0.021, percentile=2.5%)
- ops_readiness
  - image_coverage: Strong Gap (value=0.46, percentile=0.5%)
  - open_hours: Strong Gap (value=8.5, percentile=10.0%)
  - accept_time_mins: Strong Gap (value=3.2, percentile=11.0%)
- user_experience
  - prep_time_mins: Strong Gap (value=18.0, percentile=6.0%)
  - merchant_cancel_rate: Strong Gap (value=0.075, percentile=0.0%)
- supply_quality
  - active_spu_count: Strong Gap (value=22.0, percentile=3.5%)
  - discount_rate: Strong Gap (value=0.06, percentile=8.0%)

[3] Recommended Actions
- [Low] 优化商品图片或设置更有吸引力的折扣 (driver=traffic, metric=conversion_rate, score=0.34)
- [Low] 补齐菜单图片，优先补热销商品图 (driver=ops_readiness, metric=image_coverage, score=0.25)
- [Low] 延长晚间营业时间，覆盖高峰时段 (driver=ops_readiness

# 11 Analysis Plan Layer

In [17]:
from dataclasses import dataclass
from typing import List, Dict


@dataclass
class AnalysisPlan:
    intent: str
    focus_metrics: List[str]
    analysis_modules: List[str]
    time_range: str


def plan_analysis(question: str) -> AnalysisPlan:
    q = question.lower()

    # root cause
    if ("为什么" in question) or ("原因" in question) or ("下降" in question):
        return AnalysisPlan(
            intent="root_cause",
            focus_metrics=[
                "orders_7d",
                "impressions",
                "conversion_rate",
                "prep_time_mins",
                "merchant_cancel_rate",
            ],
            analysis_modules=["health", "gap"],
            time_range="7d",
        )

    # action recommendation
    if ("建议" in question) or ("怎么做" in question) or ("提升" in question):
        return AnalysisPlan(
            intent="action_recommendation",
            focus_metrics=[
                "conversion_rate",
                "image_coverage",
                "open_hours",
                "merchant_cancel_rate",
                "active_spu_count",
                "discount_rate",
            ],
            analysis_modules=["gap", "action"],
            time_range="7d",
        )

    # default: diagnosis
    return AnalysisPlan(
        intent="diagnosis",
        focus_metrics=[
            "orders_7d",
            "orders_prev_7d",
            "conversion_rate",
            "image_coverage",
            "prep_time_mins",
        ],
        analysis_modules=["health", "gap", "action"],
        time_range="7d",
    )

In [18]:
analysis_plan = plan_analysis(question)
analysis_plan

AnalysisPlan(intent='action_recommendation', focus_metrics=['conversion_rate', 'image_coverage', 'open_hours', 'merchant_cancel_rate', 'active_spu_count', 'discount_rate'], analysis_modules=['gap', 'action'], time_range='7d')

# 12 输出根据plan动态变化

In [19]:
def run_pipeline(
    merchant_row,
    question: str,
    peer_benchmark,
    health_config,
    gap_config,
):
    plan = plan_analysis(question)

    result = {
        "question": question,
        "plan": plan.__dict__,
        "merchant_id": merchant_row["merchant_id"],
        "merchant_name": merchant_row["merchant_name"],
    }

    health_result = None
    gap_results = None
    action_results = None

    if "health" in plan.analysis_modules:
        health_result = calculate_growth_health(merchant_row, health_config)
        result["health"] = health_result.__dict__

    if "gap" in plan.analysis_modules:
        gap_results = identify_gaps(merchant_row, peer_benchmark, gap_config)
        result["gaps"] = [g.__dict__ for g in gap_results]

    if "action" in plan.analysis_modules:
        if gap_results is None:
            gap_results = identify_gaps(merchant_row, peer_benchmark, gap_config)
        action_results = generate_actions(gap_results)
        result["actions"] = [a.__dict__ for a in action_results]

    return result

In [20]:
pipeline_output = run_pipeline(
    merchant_row=merchant_row,
    question=question,
    peer_benchmark=peer_benchmark,
    health_config=HEALTH_CONFIG,
    gap_config=GAP_CONFIG,
)

pipeline_output

{'question': '这家商家最近经营怎么样？我应该建议他做什么？',
 'plan': {'intent': 'action_recommendation',
  'focus_metrics': ['conversion_rate',
   'image_coverage',
   'open_hours',
   'merchant_cancel_rate',
   'active_spu_count',
   'discount_rate'],
  'analysis_modules': ['gap', 'action'],
  'time_range': '7d'},
 'merchant_id': 'M001',
 'merchant_name': 'Blue Bottle Demo Store',
 'gaps': [{'metric': 'conversion_rate',
   'driver': 'traffic',
   'percentile': 0.025,
   'value': 0.021,
   'label': 'Strong Gap'},
  {'metric': 'image_coverage',
   'driver': 'ops_readiness',
   'percentile': 0.005,
   'value': 0.46,
   'label': 'Strong Gap'},
  {'metric': 'open_hours',
   'driver': 'ops_readiness',
   'percentile': 0.1,
   'value': 8.5,
   'label': 'Strong Gap'},
  {'metric': 'accept_time_mins',
   'driver': 'ops_readiness',
   'percentile': 0.10999999999999999,
   'value': 3.2,
   'label': 'Strong Gap'},
  {'metric': 'prep_time_mins',
   'driver': 'user_experience',
   'percentile': 0.06000000000000005,
   

In [ ]:
# test


# 11 LLM Explanation Layer 

In [16]:
import tiktoken

# ===== Token Estimation =====
def estimate_text_tokens(text: str, model: str = "gpt-4.1") -> int:
    try:
        encoding = tiktoken.encoding_for_model(model)
    except KeyError:
        encoding = tiktoken.get_encoding("cl100k_base")

    return len(encoding.encode(text))


# ===== Prompt Builder =====
def build_llm_prompt(output: Dict[str, Any]) -> str:
    return f"""
你将看到一个商家经营分析系统输出的结构化结果。

请严格基于这些结果，输出以下四部分内容：

1. Executive Summary
2. Key Problems
3. Recommended Actions
4. Suggested Talking Points

要求：
- 不要编造数据
- 不要新增 action
- 用简洁中文

结构化结果：
{output}
""".strip()


# ===== Mock LLM（关键）=====
def mock_llm_response(output: Dict[str, Any]) -> str:
    return f"""
【Mock LLM Output】

该商家当前经营存在一定风险，订单表现低于历史和同类水平。

核心问题集中在转化率和供给结构不足，
同时履约效率也有一定优化空间。

建议优先优化图片与转化，同时改善出餐与取消率问题，
以提升整体经营表现。

（这是 mock 输出，未调用真实 LLM）
""".strip()


# ===== Main Function =====
def generate_llm_explanation(
    output: Dict[str, Any],
    model: str = "gpt-4.1",
    mock_mode: bool = True,   # 👈 关键开关
) -> Dict[str, Any]:

    prompt = build_llm_prompt(output)
    estimated_input_tokens = estimate_text_tokens(prompt, model=model)

    if mock_mode:
        return {
            "text": mock_llm_response(output),
            "estimated_input_tokens": estimated_input_tokens,
            "actual_input_tokens": None,
            "actual_output_tokens": None,
            "actual_total_tokens": None,
            "mode": "mock",
        }

    # ===== 真正调用 LLM（你之后再打开）=====
    from openai import OpenAI
    import os

    api_key = os.getenv("OPENAI_API_KEY")
    if not api_key:
        raise ValueError("OPENAI_API_KEY 没设置")

    client = OpenAI(api_key=api_key)

    response = client.responses.create(
        model=model,
        input=[
            {"role": "system", "content": "你是一个商家经营分析助手"},
            {"role": "user", "content": prompt},
        ],
    )

    usage = getattr(response, "usage", None)

    return {
        "text": response.output_text,
        "estimated_input_tokens": estimated_input_tokens,
        "actual_input_tokens": getattr(usage, "input_tokens", None) if usage else None,
        "actual_output_tokens": getattr(usage, "output_tokens", None) if usage else None,
        "actual_total_tokens": getattr(usage, "total_tokens", None) if usage else None,
        "mode": "real",
    }


# ===== Run =====
llm_result = generate_llm_explanation(structured_output, mock_mode=True)

print("=== LLM Summary ===")
print(llm_result["text"])
print()

print("=== Token Usage ===")
print("Estimated input tokens:", llm_result["estimated_input_tokens"])
print("Mode:", llm_result["mode"])

=== LLM Summary ===
【Mock LLM Output】

该商家当前经营存在一定风险，订单表现低于历史和同类水平。

核心问题集中在转化率和供给结构不足，
同时履约效率也有一定优化空间。

建议优先优化图片与转化，同时改善出餐与取消率问题，
以提升整体经营表现。

（这是 mock 输出，未调用真实 LLM）

=== Token Usage ===
Estimated input tokens: 837
Mode: mock


# 12 OpenAI key 

In [ ]:
# 先不要急着运行这段；等你把前面逻辑跑通再接 API

# from openai import OpenAI
# client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

# response = client.responses.create(
#     model="gpt-4.1",
#     input=build_llm_prompt(structured_output),
# )

# print(response.output_text)